In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/after_feature_selection.csv")

In [ ]:
df.shape

(9771, 26)

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

In [ ]:
df.columns

Index(['JobTitle', 'Company Size', 'company_Industry', 'Country',
       'Remote Type', 'Exp. Level', 'Years_Experience', 'Education  Level',
       'Skills - Python', 'Skills - SQL', 'Skills - ML', 'Skills-DeepLearning',
       'skills_Cloud', 'Posting Month', 'posting_Year', 'Hiring Urgency',
       'Total_Skills', 'advanced_skill_score', 'ai_specialist',
       'exp_per_skill', 'experience_intensity', 'ml_cloud_combination',
       'job_title_popularity', 'industry_frequency', 'premium_candidate',
       'Salary (USD)'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9771 entries, 0 to 9770
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   JobTitle              9771 non-null   int64  
 1   Company Size          9771 non-null   int64  
 2   company_Industry      9771 non-null   int64  
 3   Country               9771 non-null   int64  
 4   Remote Type           9771 non-null   int64  
 5   Exp. Level            9771 non-null   int64  
 6   Years_Experience      9771 non-null   float64
 7   Education  Level      9771 non-null   int64  
 8   Skills - Python       9771 non-null   int64  
 9   Skills - SQL          9771 non-null   int64  
 10  Skills - ML           9771 non-null   int64  
 11  Skills-DeepLearning   9771 non-null   int64  
 12  skills_Cloud          9771 non-null   int64  
 13  Posting Month         9771 non-null   int64  
 14  posting_Year          9771 non-null   int64  
 15  Hiring Urgency       

In [ ]:
X = df.drop(columns=['Salary (USD)'])
y = df['Salary (USD)']

In [ ]:
y_transformed = np.log1p(y)

In [ ]:
col_to_encode = ['JobTitle','company_Industry','Country','Remote Type']
num_cols = ['Years_Experience','Total_Skills','advanced_skill_score','ai_specialist','exp_per_skill','experience_intensity',
'ml_cloud_combination','job_title_popularity','industry_frequency','premium_candidate']
ordinal_cols = ['Company Size','Exp. Level','Education  Level','Hiring Urgency', ]

In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), num_cols),
        ('numerical', SimpleImputer(strategy='mean'), num_cols),
        ('cat', OneHotEncoder(drop='first'), col_to_encode),
        ('ordinal', OrdinalEncoder(), ordinal_cols)
    ],
    remainder = 'passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

In [ ]:
# K-fold cross-validation
kfold = KFold(n_splits=15, shuffle=True, random_state=12)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [ ]:
scores.mean()

np.float64(0.595801447566951)

In [ ]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    random_state = 42,
    test_size = 0.2
)

In [ ]:
# training
pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['Years_Experience',
                                                   'Total_Skills',
                                                   'advanced_skill_score',
                                                   'ai_specialist',
                                                   'exp_per_skill',
                                                   'experience_intensity',
                                                   'ml_cloud_combination',
                                                   'job_title_popularity',
                                                   'industry_frequency',
                                                   'premium_candidate']),
                                                 ('numerical', SimpleImputer(),
                                                  ['Years...
                                                   'ai_specialist',
                                                   'exp_per_skill',
                                                   'experience_intensity',
                                                   'ml_cloud_combination',
                                                   'job_title_popularity',
                                                   'industry_frequency',
                                                   'premium_candidate']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['JobTitle',
                                                   'company_Industry',
                                                   'Country', 'Remote Type']),
                                                 ('ordinal', OrdinalEncoder(),
                                                  ['Company Size', 'Exp. Level',
                                                   'Education  Level',
                                                   'Hiring Urgency'])])),
                ('model', LinearRegression())])

In [ ]:
# prediction
y_pred = pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
# evaluation
print('MAE',mean_absolute_error(y_test,y_pred))
print('R2 Score',r2_score(y_test,y_pred))


MAE 0.1414085902156313
R2 Score 0.6184443337008343


In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

models = {
    'Ridge': Ridge(random_state=42),
    'Lasso': Lasso(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(random_state=42)
}

In [ ]:
results = {}

for model_name, model in models.items():
    pipeline_model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=15, shuffle=True, random_state=12)
    scores = cross_val_score(pipeline_model, X, y_transformed, cv=kfold, scoring='r2')

    results[model_name] = scores.mean()
    print(f"{model_name} - Mean R2 Score: {scores.mean():.4f}\n")

for model_name, r2_score in results.items():
    print(f"{model_name}: {r2_score:.4f}")

Evaluating Ridge...
Ridge - Mean R2 Score: 0.5958

Evaluating Lasso...
Lasso - Mean R2 Score: 0.0079

Evaluating RandomForestRegressor...
RandomForestRegressor - Mean R2 Score: 0.7408


--- Model Comparison ---
Ridge: 0.5958
Lasso: 0.0079
RandomForestRegressor: 0.7408
